<a href="https://colab.research.google.com/github/brpetros/prompts_and_evalution_notebooks/blob/main/4_autorubric_evaluation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install krippendorff

In [ ]:
!pip install irrCAC

In [ ]:
import pandas as pd
from pprint import pprint
import numpy as np
import krippendorff
from irrCAC.raw import CAC
from sklearn.metrics import cohen_kappa_score

def calculate_reliability(file_name="evaluation_analysis.xlsx"):

    df = pd.read_excel(file_name, sheet_name='combined_evaluation')

    df = df.dropna(subset=['Manual Score', 'LLM Score'])

    results = []

    for criterion in df['Criteria'].unique():
        subset = df[df['Criteria'] == criterion]
        # Spearman Correlation
        spearman = subset['Manual Score'].corr(subset['LLM Score'], method='spearman')

        # Cohen's Kappa
        qwk = cohen_kappa_score(subset['Manual Score'].astype(int),
                                  subset['LLM Score'].astype(int), weights='quadratic')

        # Krippendorff
        data = np.array([
            subset['Manual Score'].values,
            subset['LLM Score'].values
        ])

        try:
            alpha = krippendorff.alpha(
                reliability_data=data,
                level_of_measurement='ordinal'
            )
        except Exception:
            alpha = np.nan

        # AC2
        cac = CAC(subset[['Manual Score', 'LLM Score']])

        # Gwet's AC1
        gwet_res = cac.gwet()

        # percentage of agreement
        agreement = (subset['Manual Score'] == subset['LLM Score']).mean() * 100

        results.append({
            'Criterion': criterion,
            'Spearman_Correlation': round(spearman,2),
            'QWK': round(qwk,2),
            'Krippendorff_Alpha': round(alpha,2),
            f'{gwet_res['est']['coefficient_name']}': round(gwet_res['est']['coefficient_value'],2),
            'aggreement %': round(agreement,2)
        })

    return pd.DataFrame(results)

irr_df = calculate_reliability()
print(irr_df)
irr_df.to_excel("reliability_analysis.xlsx")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pprint import pprint

def calculate_discriminant_validity(file_name="evaluation_analysis.xlsx"):

    df = pd.read_excel(file_name, sheet_name='combined_evaluation')

    llm_pivot = df.pivot_table(index=['Prompt Type','Model Name','Skill Name'],
                               columns='Criteria',
                               values='LLM Score')

    human_pivot = df.pivot_table(index=['Prompt Type','Model Name','Skill Name'],
                                 columns='Criteria',
                                 values='Manual Score')

    llm_corr = llm_pivot.corr(method='pearson')
    human_corr = human_pivot.corr(method='pearson')

    print("--- LLM Inter-Criteria Correlation (Pearson) ---")
    print(llm_corr.round(2))
    print("\n--- Human Inter-Criteria Correlation (Pearson) ---")
    print(human_corr.round(2))

    with pd.ExcelWriter("discriminant_validity_analysis.xlsx") as writer:
        llm_corr.to_excel(writer, sheet_name='LLM_Correlation')
        human_corr.to_excel(writer, sheet_name='Human_Correlation')

    # Heatmap to visualize
    plt.figure(figsize=(10, 8))
    sns.heatmap(llm_corr, annot=True, cmap='Reds', vmin=0, vmax=1)
    plt.savefig("llm_halo_effect_heatmap.png")

    return llm_corr, human_corr

llm_matrix, human_matrix = calculate_discriminant_validity()

In [ ]:

plt.figure(figsize=(10, 8))
sns.heatmap(human_matrix, annot=True, cmap='Reds', vmin=0, vmax=1)
plt.savefig("human_heatmap.png")